# Kifaru — poaching detection model

Fine-tunes a YOLO nano model for the aerial surveillance MVP and exports it to
ONNX, which is what the FastAPI service loads in production.

**Run this on Colab with a GPU runtime** (`Runtime → Change runtime type → T4 GPU`).
The deployment server has no GPU, so training here and shipping weights is the
only workable split.

### Why the class set is what it is

A rhino in frame is not poaching. A *person* or *vehicle* near a rhino is — and
closing on one is more so. The model detects objects only; the distance and
approach logic lives above it in `app/geometry.py` and `app/threat.py`.

| id | class | why |
|----|---------|-----------------------------------------|
| 0 | rhino | the asset being protected — also the **ruler** the scorer measures distance with |
| 1 | person | primary threat indicator |
| 2 | vehicle | car/truck/motorcycle, collapsed |
| 3 | weapon | *optional* — escalates an alert, never required to raise one |

Class 3 is in `app/config.py` but is **not** trained here, because there is no
weapon data yet. That is safe: the server reads class names from the ONNX
metadata, so a 3-class model simply never emits `weapon`. Add it to `CLASSES`
below only once you have labelled examples — a class with zero instances just
wastes capacity.

> **The trap this avoids:** fine-tuning on rhino-only images rebuilds the
> detection head for a single class, and the model permanently loses COCO's
> `person` / `car` / `truck`. It would then be structurally incapable of
> detecting poaching. Keep people and vehicles in the label set.

> **Why rhino recall matters twice.** Miss the rhino and you do not merely lose
> one box — the scorer loses its ruler, every distance becomes unmeasurable,
> and the frame falls back to a low baseline score. Rhino recall is effectively
> a floor on the whole system.

## 1. Setup

Versions are pinned deliberately — an unpinned reinstall weeks from now can
shift training defaults and make results non-reproducible.

**Run this cell first and read its output.** It prints whether your code is
executing on a Colab VM or on your own machine, which determines everything
else: where files land, whether Drive is needed, and whether cloning the repo
is necessary at all. It only clones when the repo is not already on disk, so a
setup that already has the checkout needs no GitHub token.

Export and model-checking are **not** reimplemented in this notebook; they live
in `training/export_onnx.py` and `training/check_model.py` and are called from
here, so the opset, image size and output-layout checks cannot drift from what
the server actually does.

In [ ]:
!pip install -q ultralytics==8.4.118 onnxruntime==1.20.1 onnx onnxslim python-dotenv

import shutil
import subprocess
from pathlib import Path

import ultralytics

ultralytics.checks()

REPO_SLUG = "sip-project-group-15/api"

try:
    import google.colab  # noqa: F401

    ON_COLAB_VM = True
except ImportError:
    ON_COLAB_VM = False

print(
    "runtime:  Colab VM — this filesystem is Google's, not your laptop's"
    if ON_COLAB_VM
    else "runtime:  local — this filesystem IS your laptop's"
)
print("cwd:     ", Path.cwd())


def git(*args) -> subprocess.CompletedProcess:
    return subprocess.run(["git", *args], capture_output=True, text=True)


def find_repo(start: Path) -> Path | None:
    """Find a checkout we are already running inside.

    A local runtime, or an editor that syncs your workspace onto the runtime,
    means the repo is already on disk. Cloning again would shadow the very
    edits you are trying to test, and would need credentials for no reason.
    """
    for candidate in (start, *start.parents):
        if (candidate / "training" / "export_onnx.py").is_file():
            return candidate
    return None


# Everything this notebook calls out to. Checked explicitly, because a clone
# can succeed and still land a commit that predates this code.
REQUIRED = (
    "training/export_onnx.py",
    "training/check_model.py",
    "app/detector.py",
    "app/threat.py",
)


def is_checkout(path: Path) -> bool:
    """Whether this is a git checkout rather than the debris of a failed clone.

    An interrupted or rejected clone leaves its target directory behind. Mere
    existence is not evidence of anything, and treating it as a checkout turns
    a clear failure here into a confusing one several cells later.
    """
    return (path / ".git").is_dir()


def missing_pieces(path: Path) -> list[str]:
    """Expected files absent from a checkout, i.e. work that was never pushed."""
    return [name for name in REQUIRED if not (path / name).exists()]


def read_token() -> tuple[str | None, str]:
    """Read GH_TOKEN from Colab secrets, naming the failure if there is one.

    Only reached when an anonymous clone has already failed. Note that secrets
    are readable *only from the Colab web UI* — through the VS Code extension
    or any other client this times out, so a private repo cannot be cloned
    that way at all.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return None, "not running on Colab"

    try:
        return userdata.get("GH_TOKEN"), "found"
    except Exception as error:
        name = type(error).__name__
        if "Timeout" in name:
            return None, (
                "Colab secrets are readable only from the Colab web UI, not "
                "the VS Code extension. Either make the repo public, or run "
                "this notebook at colab.research.google.com"
            )
        if "NotebookAccess" in name:
            return None, (
                "GH_TOKEN exists, but this notebook is not authorised to read "
                "it — open the key icon in the sidebar and enable notebook "
                "access for it"
            )
        if "SecretNotFound" in name:
            return None, "no Colab secret named GH_TOKEN"
        return None, f"could not read GH_TOKEN ({name}: {error})"


def clone(destination: Path) -> None:
    """Clone anonymously; only reach for a token if that is refused.

    Trying the token first would cost a ten-second secret-fetch timeout on
    every run of a public repo, for nothing.
    """
    result = git("clone", "-q", f"https://github.com/{REPO_SLUG}.git", str(destination))
    if not result.returncode:
        print("repo:     cloned ->", destination)
        return

    anonymous_error = result.stderr.strip()
    token, token_status = read_token()
    print("token:   ", token_status)

    if not token:
        raise SystemExit(
            f"git clone failed: {anonymous_error}\n\n"
            f"If {REPO_SLUG} is private, it cannot be cloned without "
            "credentials.\nThe simplest fix for a student project is to make "
            "the repo public.\nOtherwise run this notebook in the Colab web "
            "UI and add a fine-grained\ntoken as a secret named GH_TOKEN "
            "(Contents: Read-only)."
        )

    shutil.rmtree(destination, ignore_errors=True)
    result = git(
        "clone", "-q", f"https://{token}@github.com/{REPO_SLUG}.git", str(destination)
    )
    if result.returncode:
        # git echoes the remote URL on failure and that URL carries the token,
        # so it must never reach the notebook output.
        raise SystemExit(
            "git clone failed even with a token: "
            f"{result.stderr.strip().replace(token, '***')}\n\n"
            f"The token is likely expired or lacks Contents=Read on {REPO_SLUG}."
        )

    print("repo:     cloned with token ->", destination)


REPO_DIR = find_repo(Path.cwd())

if REPO_DIR:
    print("repo:     already here, no clone needed ->", REPO_DIR)
else:
    REPO_DIR = Path("/content/api")

    if REPO_DIR.exists() and not is_checkout(REPO_DIR):
        print("repo:     discarding incomplete checkout at", REPO_DIR)
        shutil.rmtree(REPO_DIR, ignore_errors=True)

    if is_checkout(REPO_DIR):
        result = git("-C", str(REPO_DIR), "pull", "--ff-only", "-q")
        if result.returncode:
            # Do not report success on a failed update: every later cell would
            # then silently run against stale code.
            raise SystemExit(f"git pull failed: {result.stderr.strip()}")
        print("repo:     updated ->", REPO_DIR)
    else:
        clone(REPO_DIR)

missing = missing_pieces(REPO_DIR)
if missing:
    listed = "\n".join(f"  - {name}" for name in missing)
    raise SystemExit(
        f"The checkout at {REPO_DIR} is missing:\n{listed}\n\n"
        "The clone itself worked — this commit simply predates that code, so it\n"
        "has not been pushed yet. From the repo on your machine:\n\n"
        "    git add -A\n"
        '    git commit -m "add distance scoring and training pipeline"\n'
        "    git push\n\n"
        "then re-run this cell. Colab clones from GitHub, so anything that is\n"
        "only on your laptop is invisible here."
    )

# Confirm which commit is about to be trained from; if this is not the work you
# pushed, the export and scoring code here is not the code you just edited.
print("commit:  ", git("-C", str(REPO_DIR), "log", "-1", "--format=%h %s").stdout.strip())

## 2. Mount Drive

Colab wipes `/content` when the session ends (~90 min idle, 12h hard cap). A
long training run that writes only to `/content` loses `best.pt` when the tab
disconnects. Everything below writes to Drive instead.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/kifaru")
RUNS_DIR = PROJECT_DIR / "runs"
WEIGHTS_DIR = PROJECT_DIR / "weights"
DATA_DIR = Path("/content/datasets")  # scratch: large, re-downloadable

for directory in (RUNS_DIR, WEIGHTS_DIR, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("checkpoints ->", RUNS_DIR)

## 3. Config

`imgsz=640` matches what the production ONNX graph will be frozen at — change it
here and the server-side preprocessing must change with it.

In [ ]:
CONFIG = {
    "base_model": "yolo26n.pt",  # nano: the only sane size for a 6-core CPU server
    "epochs": 100,               # with patience below, this stops early on plateau
    "patience": 20,
    "imgsz": 640,
    "batch": 16,
    "seed": 0,
}

CLASSES = {0: "rhino", 1: "person", 2: "vehicle"}

CONFIG

## 4. Inspect the base dataset

Confirm African Wildlife actually contains a rhino class before building
anything on top of it. This downloads ~1.5k images on first run.

In [ ]:
from ultralytics.data.utils import check_det_dataset

wildlife = check_det_dataset("african-wildlife.yaml")

print("classes:", wildlife["names"])
print("root:   ", wildlife["path"])

RHINO_ID = next(
    (i for i, name in wildlife["names"].items() if "rhino" in name.lower()), None
)
assert RHINO_ID is not None, "No rhino class — pick a different base dataset."
print(f"\nrhino is class {RHINO_ID} in the source dataset")

## 5. Build the merged dataset

African Wildlife supplies rhinos but **no people or vehicles**, so it cannot
train the poaching signal on its own. This cell remaps its rhino labels into our
class set and leaves a clearly marked slot for the threat imagery.

### What you must supply

Drop YOLO-format data into `drive/MyDrive/kifaru/custom/` as
`images/{train,val}` + `labels/{train,val}`, using **our** class ids (0/1/2).
Two sources worth combining:

- **`VisDrone.yaml`** (built into Ultralytics) — real aerial imagery with
  pedestrian/car/van/truck. Right viewpoint for the threat classes.
- **Your own drone footage**, annotated in Roboflow or CVAT. This is the only
  source of *rhinos seen from above*, and it is what will actually decide
  whether the model works in the field.

> **Domain gap warning:** African Wildlife is ground-level photography. A model
> trained solely on it will underperform badly on top-down drone frames —
> different silhouette, scale and background. Treat it as a starting point, not
> a finished training set.

In [ ]:
import shutil

MERGED = DATA_DIR / "kifaru-merged"
CUSTOM = PROJECT_DIR / "custom"  # your own annotated data, already in 0/1/2

# Source class id -> our class id. Anything absent is dropped, so buffalo,
# elephant and zebra fall away; add them here if you want them detected too.
REMAP = {RHINO_ID: 0}


def convert_split(src_root: Path, split: str, remap: dict[int, int]) -> int:
    """Copy one split into MERGED, rewriting label ids and dropping the rest."""
    src_images = src_root / "images" / split
    src_labels = src_root / "labels" / split
    if not src_images.is_dir():
        print(f"  (no {split} split at {src_images})")
        return 0

    dst_images = MERGED / "images" / split
    dst_labels = MERGED / "labels" / split
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    kept = 0
    for image_path in src_images.iterdir():
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        label_path = src_labels / f"{image_path.stem}.txt"
        lines = []
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.split()
                if not parts:
                    continue
                source_id = int(parts[0])
                if source_id in remap:
                    lines.append(" ".join([str(remap[source_id]), *parts[1:]]))

        # Images with no surviving boxes are still useful: they teach the model
        # what a non-rhino scene looks like and suppress false positives.
        shutil.copy2(image_path, dst_images / image_path.name)
        (dst_labels / f"{image_path.stem}.txt").write_text(
            "\n".join(lines) + ("\n" if lines else "")
        )
        kept += len(lines)
    return kept


shutil.rmtree(MERGED, ignore_errors=True)
wildlife_root = Path(wildlife["path"])

print("African Wildlife -> merged:")
for split in ("train", "val"):
    print(f"  {split}: {convert_split(wildlife_root, split, REMAP)} rhino boxes")

if CUSTOM.is_dir():
    print("\nYour custom data -> merged:")
    for split in ("train", "val"):
        # Already in our id space, so the remap is an identity mapping.
        n = convert_split(CUSTOM, split, {i: i for i in CLASSES})
        print(f"  {split}: {n} boxes")
else:
    print(f"\n!! No custom data at {CUSTOM}")
    print("!! Training will see ZERO person/vehicle examples, so those classes")
    print("!! cannot be learned. Fine for a first pipeline run; not shippable.")

In [ ]:
import yaml

data_yaml = MERGED / "kifaru.yaml"
data_yaml.write_text(
    yaml.safe_dump(
        {
            "path": str(MERGED),
            "train": "images/train",
            "val": "images/val",
            "names": CLASSES,
        },
        sort_keys=False,
    )
)
print(data_yaml.read_text())

### Class balance

Check this before training. If one class has an order of magnitude fewer boxes
than another, the model will largely ignore it and mAP will still look fine.

In [ ]:
from collections import Counter

for split in ("train", "val"):
    counts = Counter()
    label_dir = MERGED / "labels" / split
    if not label_dir.is_dir():
        continue
    for label_file in label_dir.glob("*.txt"):
        for line in label_file.read_text().splitlines():
            if line.strip():
                counts[CLASSES[int(line.split()[0])]] += 1
    total = sum(counts.values()) or 1
    print(f"{split}: {dict(counts)}")
    for name, n in counts.items():
        print(f"    {name:<8} {n:>6}  ({100 * n / total:.1f}%)")

## 6. Train

Resumable: if the Colab session drops, re-running picks up from the last Drive
checkpoint rather than starting over.

In [ ]:
from ultralytics import YOLO

RUN_NAME = "kifaru-v1"
last_checkpoint = RUNS_DIR / RUN_NAME / "weights" / "last.pt"

if last_checkpoint.exists():
    print(f"Resuming from {last_checkpoint}")
    model = YOLO(str(last_checkpoint))
    results = model.train(resume=True)
else:
    model = YOLO(CONFIG["base_model"])
    results = model.train(
        data=str(data_yaml),
        epochs=CONFIG["epochs"],
        patience=CONFIG["patience"],
        imgsz=CONFIG["imgsz"],
        batch=CONFIG["batch"],
        seed=CONFIG["seed"],
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        plots=True,
    )

## 7. Validate

Your current notebook has no metrics at all, so there is no way to tell whether
a retrain helped. Record these numbers for every run.

`mAP50-95` is the headline. **Per-class recall matters more here** — a missed
poacher is far worse than a false alarm a ranger dismisses.

In [ ]:
best_weights = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
model = YOLO(str(best_weights))
metrics = model.val(data=str(data_yaml), imgsz=CONFIG["imgsz"])

print(f"\nmAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}\n")

for i, name in CLASSES.items():
    try:
        p, r, ap50, ap = metrics.box.class_result(i)
        print(f"{name:<8} precision={p:.3f} recall={r:.3f} mAP50={ap50:.3f}")
    except (IndexError, KeyError):
        print(f"{name:<8} — no validation examples")

## 8. Export to ONNX

**What training produced is `best.pt`.** That checkpoint is the model — it is
what accumulates the training, what a future run resumes from, and the only
thing a re-export can start from. ONNX is a one-way build artifact frozen out
of it for serving: you cannot train it, and nothing can recover a `.pt` from an
`.onnx`. So `best.pt` goes to Drive and stays there.

The production image serves the ONNX rather than PyTorch. On a 6-core shared
CPU that is the difference between a ~250MB image starting in ~1s and a ~2GB
image starting in 10–20s, with faster inference besides.

The export itself is **not written here** — it is `training/export_onnx.py`,
called below. That script also verifies the result: it loads the graph through
onnxruntime exactly as the server will, and checks the output layout using
`app/detector.py`'s own predicate, so a graph the server cannot parse fails
here instead of silently producing wrong boxes in production.

In [ ]:
destination = WEIGHTS_DIR / "best.onnx"
IMGSZ = CONFIG["imgsz"]

# The checkpoint first: it is the only artifact a future re-export or a resumed
# training run can start from, and /content is wiped when the session ends.
shutil.copy2(best_weights, WEIGHTS_DIR / "best.pt")

!python {REPO_DIR}/training/export_onnx.py "{best_weights}" --destination "{destination}" --imgsz {IMGSZ}

## 9. Rough CPU timing

This measures raw graph throughput on a blank tensor — an upper bound on how
fast the server could go, ignoring video decoding and scoring. Colab's CPU is a
reasonable stand-in; multiply by ~1.5–2× for contention, since the deployment
box shares 6 cores across containers.

Use the per-frame figure to sanity-check the sampling rate. At 30fps, analysing
every frame of a 5-minute clip is 9,000 inferences — far past the 600s nginx
timeout. Sampling ~2 frames/second cuts that 15× and loses nothing
operationally, since poaching activity does not vanish within 500ms.

For the real end-to-end cost on actual footage, run `training/check_model.py`
against a clip from the repo — it times the same detector the API serves, with
decoding and scoring included.

In [ ]:
import time

import numpy as np
import onnxruntime as ort

session = ort.InferenceSession(str(destination), providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
dummy = np.zeros((1, 3, IMGSZ, IMGSZ), dtype=np.float32)

for _ in range(3):  # warm up
    session.run(None, {input_name: dummy})

start = time.perf_counter()
runs = 20
for _ in range(runs):
    session.run(None, {input_name: dummy})
per_frame = (time.perf_counter() - start) / runs

print(f"{per_frame * 1000:.1f} ms/frame on Colab CPU")
print(f"~{per_frame * 2000:.0f} ms/frame estimated on the server\n")
for minutes in (1, 5):
    frames = minutes * 60 * 2  # sampling at 2 fps
    print(f"{minutes}min clip @2fps = {frames} frames ~ {frames * per_frame * 2:.0f}s")

## Next steps

1. Download `best.onnx` from `Drive/MyDrive/kifaru/weights/` into the API repo at
   `models/best.onnx`.
2. Check it before trusting it, from the repo root:

   ```bash
   python training/check_model.py some_clip.mp4 --annotate out/
   ```

   That runs the real detector, tracker and scorer, prints the score breakdown
   per frame, and estimates per-frame cost on the server. Look at `out/` — if
   the boxes are in the wrong place, the export layout is being misread and no
   amount of scorer tuning will help.
3. Commit `models/best.onnx` and push to `main` to deploy.

### Before you have a trained model

You can exercise the entire pipeline today using stock COCO weights, which
already detect `person` and vehicles well:

```bash
python training/export_onnx.py yolo26n.pt --destination models/baseline-coco.onnx
python training/check_model.py clip.mp4 --model models/baseline-coco.onnx \
    --aliases "car=vehicle,truck=vehicle,bus=vehicle,elephant=rhino"
```

COCO has no rhino, so aliasing an animal it *does* know stands in for one and
lets the distance and approach logic be validated on real footage before a
single epoch of training. `models/baseline-*.onnx` is gitignored — it knows
nothing about rhinos and must never reach a deploy.

### Recording results

Keep the per-class precision/recall from step 7 for every run. Detector mAP is
only half the evaluation: the scorer needs its own scenario-level test set —
20–30 clips labelled poaching / benign, including hard negatives like tourists
and rangers near rhinos — measured as alert precision and recall. The two
numbers move independently, and only the second one is what the demo shows.

**Before committing this notebook:** `Edit → Clear all outputs`. Training logs
and embedded plot images make diffs unreadable and bloat the repo.